Langchain, Huggignface Models are Used with Native LLMChain (SequenceChain).<br/>

Chatbot - Ex: Travel Assistant. <br/>
Model used - TinyLlama/TinyLlama-1.1B-Chat-v1.0 <br/>
Chat History is chained to give a Convrsational feeling and History Content for the Assistant.<br/>

Output of Chatbot is Translated to French. <br/>
Model used - Helsinki-NLP/opus-mt-en-fr <br/>
Translation is only to print the output.

All this individual LLM activity as well as custom pre/post parsing funcation are wired using SequenceChain

In [5]:
# Import PythinNative packages
import os
import keyboard
from dotenv import load_dotenv

# Import LangChain and Huggingface Chatbot Interface
from langchain_huggingface import ChatHuggingFace
from langchain_huggingface import HuggingFacePipeline
from langchain_core.runnables import RunnableLambda, RunnableSequence
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv()

True

In [6]:
# Instantiate ChatHuggignFace LLM for Chatbot 
hf_chat_llm = HuggingFacePipeline.from_model_id(
              model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
              task="text-generation",
              device=-1,
              pipeline_kwargs={
                  "max_new_tokens":1000,
                  "temperature":0.9,
                  "top_p":0.8
              }
            )

chatbot = ChatHuggingFace(llm=hf_chat_llm)

c:\Products\Anaconda3\envs\hf1_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


In [7]:
# Instantiate Tranlslation LLM for Chatbot.
hf_trans_en_fr_llm = HuggingFacePipeline.from_model_id(
                   model_id = "Helsinki-NLP/opus-mt-en-fr",
                   task="translation",
                   device=-1,
                   pipeline_kwargs={
                     "max_new_tokens": 1000
                  }
                )

c:\Products\Anaconda3\envs\hf1_env\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


In [8]:
# Format Chat Chain

def format_system_messages(chat_hist, sys_msg):
    chat_hist = [
        SystemMessage(content=sys_msg)
    ]
    return(chat_hist)

def format_user_messages(chat_hist, usr_msg):
    chat_hist.append(HumanMessage(content=usr_msg))
    return(chat_hist)

def format_assistant_messages(chat_hist, ai_msg):
    chat_hist.append(AIMessage(content=ai_msg))
    return(chat_hist)

In [9]:
#Chat with AIAssistant

def chat(chat_hist):
    response = chatbot.invoke(chat_hist)
    print(response.content)
    ai_message = response.content.split("<|assistant|>")[-1].strip()
    return(ai_message)

In [17]:
# Translate Chat to French 
# Line by line of respnse sentece from LLM. 

def translate(trans_input):
    sentence_in = []
    sentence_out = []
    sentence_in = trans_input.splitlines()     # Split Sentence on '\n'
    
    for text in sentence_in:
        if (text != ''):
            text_out = hf_trans_en_fr_llm.invoke(text)
            sentence_out.append(text_out)

    trans_out = ''
    for text in sentence_out:                  # Concatinate Array with '\n'
        trans_out = trans_out + text + '\n\n'

    print('French Translation\n', trans_out)
    return(trans_out)

In [18]:
# Initiate Chat with AI

chat_prompt = []
exit_flag = False

def on_key_press(event):
    if event.name == 'esc':
        print("Escape key pressed! Exiting input.")
        global exit_flag
        exit_flag = True
        return True  # Stop the keyboard listener

# Keyboard Listener
keyboard.on_press(on_key_press)

# System Prompt - Set teh Context for the bot.
system_prompt = input("What type Assistant should i be today: ")
chat_prompt = format_system_messages(chat_prompt, system_prompt)

#Converting Custom code to RunnableSequence
run_update_usrmessage = RunnableLambda(lambda usr_input: format_user_messages(usr_input['chat_prompt'], usr_input['user_prompt']))
run_update_aimessage = RunnableLambda(lambda ai_input: format_assistant_messages(ai_input['chat_prompt'], ai_input['ai_message']))
run_chat = RunnableLambda(chat)
run_translate = RunnableLambda(translate)

# Run LangChain Sequence
chain_activity = (
      run_update_usrmessage
    | run_chat 
    | run_translate
    | (lambda ai_resp: {'chat_prompt': chat_prompt, 'ai_message': ai_resp})
    | run_update_aimessage
)

while(not exit_flag):
    user_prompt = input("User: ")
    if (not exit_flag):
        chain_activity.invoke({'chat_prompt': chat_prompt, 'user_prompt':  user_prompt})

<|system|>
You are a helpful Travel Assistant</s>
<|user|>
Information about top 10 things to see in Spain</s>
<|assistant|>
1. The Alhambra: Located in Granada, Spain, the Alhambra is a UNESCO World Heritage Site. The palace and fortress was built by the Moors and features stunning Arabic architecture.

2. The Roman Theatre of Mérida: The Roman Theatre of Mérida was built in the 1st century AD and is one of the largest and most well-preserved theatres in the Roman Empire.

3. The Sagrada Familia: This iconic building is a masterpiece of Gaudi's architecture. It is a stunning example of Catalan Gothic style and has been under construction since 1882.

4. The Gothic Quarter: Located in Barcelona, Spain, the Gothic Quarter is a historic neighborhood that features medieval architecture, narrow streets, and plenty of shopping opportunities.

5. The San Juan de Dios Park: Located in Seville, Spain, the San Juan de Dios Park is a public park with a beautiful fountain, gardens, and a museum.


In [ ]:
        #chat_chain = format_user_messages(chat_chain, user_prompt)
        # Convert to LLM Chain
        #ai_response = chat(chat_chain)
        #fr_response = translate(ai_response) 
        #print("Assistant French:\n", fr_response)
        #chat_chain = format_assistant_messages(chat_chain, ai_response)

#run_chat = RunnableLambda(lambda chat_input: chat(chat_input['chat_prompt']))
#run_translate = RunnableLambda(lambda input: translate(input['trans_input']))